<a href="https://colab.research.google.com/github/naunauwf/satria-data-2026/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
# sel 1: mount content drive (satria_data_2026)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# sel 2: import library
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from cv2 import imread, cvtColor, COLOR_BGR2RGB

In [ ]:
# sel 3: untuk menghubungkan albumentations dengan ImageFolder
class AlbumentationsWrapper:
  def __init__(self, transform):
    self.transform = transform

  def __call__(self, img):
    # mngubah format img PIL dari ImageFolder menjadi numpy array agar bisa dibaca Albumentations
    img_np = np.array(img)
    augmented = self.transform(image=img_np)
    return augmented['image']

In [ ]:
# sel 4: konfigurasi jalur folder (path) & transformasi gambar
TRAIN_DIR = "/content/drive/MyDrive/Satria_Data_2026/dataset/train"
TEST_DIR = "/content/drive/MyDrive/Satria_Data_2026/dataset/test"
SUBMISSION_TEMPLATE_PATH = "/content/drive/MyDrive/Satria_Data_2026/submission.csv"
MODEL_SAVE_PATH = "/content/drive/MyDrive/Satria_Data_2026/checkpoints/efficientnet_baseline.pth"
OUTPUT_PATH = "/content/drive/MyDrive/Satria_Data_2026/output/submission_ourti24f.csv"

In [ ]:
# sel 5: training data transformation (dengan augemntasi)
train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

In [ ]:
# sel 6: training data test (tanpa augmentasi acak)
val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

In [ ]:
# sel 7: memuat datasets ke dataloader

# wrapped transform function
wrapped_train_transform = AlbumentationsWrapper(train_transform)

# membaca data scra otomatis berdasarkan struktur folder (0_Recyclable, 1_Electronic, 2_Organic)
train_dataset = ImageFolder(root=TRAIN_DIR, transform=wrapped_train_transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)

print("Kategori kelas yang terdeteksi:", train_dataset.class_to_idx)
print("Total jumlah gambar latihan:", len(train_dataset))

Kategori kelas yang terdeteksi: {'0_Recyclable': 0, '1_Electronic': 1, '2_Organic': 2}
Total jumlah gambar latihan: 26527


In [ ]:
# sel 8: inisialisasi model, loss function, and optimizer

# detect GPU/CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Perangkat yang digunakan:", device)

# memanggil model pretrained dan mengubah layer terakhir ke 3 kelas output
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=3)
model = model.to(device)

# penentu loss function and algorithm perbaikan model
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

Perangkat yang digunakan: cpu


In [ ]:
# sel 9: proses training model (training loop)

EPOCHS=5

for epoch in range(EPOCHS):
  model.train()
  running_loss = 0.0
  correct = 0
  total = 0

  for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    running_loss += loss.item() * images.size(0)
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

  epoch_loss = running_loss / len(train_dataset)
  epoch_acc = (correct / total) * 100

  print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {epoch_loss:.4f} - Akurasi Train: {epoch_acc:.2f}%")

print("Proses pelatihan model selesai.")

# save bobot model file ke gdrive
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"File bobot (.pth) berhasil disimpan di: {MODEL_SAVE_PATH}")


Epoch [1/5] - Loss: 0.4706 - Akurasi Train: 87.99%
Epoch [2/5] - Loss: 0.1282 - Akurasi Train: 95.54%
Epoch [3/5] - Loss: 0.0781 - Akurasi Train: 97.30%
Epoch [4/5] - Loss: 0.0509 - Akurasi Train: 98.24%
Epoch [5/5] - Loss: 0.0465 - Akurasi Train: 98.40%
Proses pelatihan model selesai.
File bobot (.pth) berhasil disimpan di: /content/drive/MyDrive/Satria_Data_2026/checkpoints/efficientnet_baseline.pth


In [17]:
# sel 10: prediksi data uji & pembuatan file submission.csv

df_sub = pd.read_csv(SUBMISSION_TEMPLATE_PATH)

# change model to evaluation mode
model.eval()
predictions = []

print("memulai proses inferensi otomatis pada data uji")
with torch.no_grad():
  for idx, row in df_sub.iterrows():
    img_id = int(row['id'])

    # cek apkh extnsn. .jpg or .jpeg
    img_path = os.path.join(TEST_DIR, f"{img_id}.jpg")
    if not os.path.exists(img_path):
      img_path = os.path.join(TEST_DIR, f"{img_id}.jpeg")

    # jika keduanya tidak ketemu, berikan fallback security agar loop tidak berhenti
    if not os.path.exists(img_path):
      print(f"warning: file gambar untuk ID {img_id} not found.")
      predictions.append(0) # isi default bebas agar baris csv tidak bergeser
      continue

    # read img dan sesuaikan warna
    image = imread(img_path)
    image = cvtColor(image, COLOR_BGR2RGB)

    # transformasi validasi (tanpa augmentasi acak)
    transformed = val_transform(image=image)['image']
    tensor_img = transformed.unsqueeze(0).to(device)

    # execution tebakan model
    output = model(tensor_img)
    pred_class = torch.argmax(output, dim=1).item()

    predictions.append(pred_class)

# msukan daftar hasil tebakan ke kolom 'predicted
df_sub['predicted'] = predictions

# save results akhir ke folder output
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df_sub.to_csv(OUTPUT_PATH, index=False)
print(f"Proses selesai. Berkas siap unggah berada di: {OUTPUT_PATH}")


memulai proses inferensi otomatis pada data uji
Proses selesai. Berkas siap unggah berada di: /content/drive/MyDrive/Satria_Data_2026/output/submission_ourti24f.csv
